In [45]:
import json
import os
import tempfile
from sklearn.metrics import accuracy_score, classification_report
import random
import requests
import statistics

In [ ]:
def build_prompt(paper):
  metadata = paper.get('metadata') #metadata dictionary that contains the actual contents of the paper

  soundness_guideline = f"""
  First, is the technical approach sound and well-chosen? 
  Second, can one trust the empirical claims of the paper -- are they supported by proper experiments and are the results of the experiments correctly interpreted?  
  5 = The approach is very apt, and the claims are convincingly supported. 
  4 = Generally solid work, although there are some aspects of the approach or evaluation I am not sure about. 
  3 = Fairly reasonable work. The approach is not bad, and at least the main claims are probably correct, but I am not entirely ready to accept them (based on the material in the paper). 
  2 = Troublesome. There are some ideas worth salvaging here, but the work should really have been done or evaluated differently. 
  1 = Fatally flawed.
  """
  content_list = metadata.get('sections')
  #print(content_list)
  #print(content_list[4].get('text'))
  #print(type(content_list[4]))
  prompt_soundness = f""" You are a reviewer for an NLP academic conference and are critiquing this paper for its SOUNDNESS. Your task is to identify any issues and cite the 
  specific heading where the issue arises. 

  Paper Contents: {str(metadata.get('sections'))}

  Consider SOUNDNESS as folllows: {soundness_guideline} 

  Using the paper contents and the definition of SOUNDNESS, score the paper on SOUNDNESS, provide your justificaiton, and evidence with reference to headings.
  
  """
  #prompt = "give me three animals in the prediction, reasoning, evidence fields" did this to see if qwen would respond to something simple bc at first it returned nothing

  prompt_soundness_section = f""" You are a reviewer for an NLP academic conference and are critiquing this paper for its SOUNDNESS. Your task is to identify any issues and cite the 
  specific heading where the issue arises. 

  Paper Contents: {content_list[4].get('text')}

  Consider SOUNDNESS as folllows: {soundness_guideline} 

  Using the paper contents and the definition of SOUNDNESS, score the paper on SOUNDNESS, provide your justification, and evidence with reference to headings.

  """

  
  prompt_soundness_section_2 = f""" You are helping me write a submission for an NLP academic conference and are critiquing this paper for its SOUNDNESS. Your task is to identify any issues and cite the 
  specific heading where the issue arises. Be as critical and specific as possible.

  Paper Contents: {content_list[4].get('text')}

  Consider SOUNDNESS as folllows: {soundness_guideline} 

  Using the paper contents and the definition of SOUNDNESS, score the paper on SOUNDNESS, provide your justification, and evidence with reference to headings.

  """


  prompt_soundness_as_writing_assistant = f""" You are critiquing this paper for its SOUNDNESS. Your task is to identify any issues and cite the 
  specific heading where the issue arises. Be as critical and specific as possible.

  Paper Contents: {content_list[4].get('text')}

  Consider SOUNDNESS as folllows: {soundness_guideline} 

  Using the paper contents and the definition of SOUNDNESS, score the paper on SOUNDNESS, provide your justification, and evidence with reference to headings.

  """


  return prompt_soundness_section_2


In [46]:
pdf_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\acl_2017\\train\\parsed_pdfs\\699.pdf.json"
with open(pdf_path, 'r') as f1:
    paper = json.load(f1) #json file contents for one research paper

build_prompt(paper)

[{'heading': None, 'text': '1 000\n011\n012\n013\n014\n015\n016\n017\n018\n019\n020\n021\n022\n023\n024\n025\n026\n027\n028\n029\n030\n031\n032\n033\n034\n035\n036\n037\n038\n039\n040\n041\n042\n043\n044\n045\n046\n047\n048\n049\n061\n062\n063\n064\n065\n066\n067\n068\n069\n070\n071\n072\n073\n074\n075\n076\n077\n078\n079\n080\n081\n082\n083\n084\n085\n086\n087\n088\n089\n090\n091\n092\n093\n094\n095\n096\n097\n098\n099'}, {'heading': '1 Introduction', 'text': 'Keyphrase or keyword is a piece of short and summative content that expresses the main semantic meaning of a long text. A typical use of keyphrase or keyword is in scientific publications, to provide the core information of a paper. We use the term “keyphrase”, interchangeable as “keyword”, in the rest of this paper, as it implies that it may contain multiple words. High-quality keyphrases can facilitate the understanding, organizing and accessing of document content. As a result, many stud-\nies have devoted to studying the way

" You are a reviewer for an NLP academic conference and are critiquing this paper for its SOUNDNESS. Your task is to identify any issues and cite the \n  specific heading where the issue arises. \n\n  Paper Contents: [{'heading': None, 'text': '1 000\\n011\\n012\\n013\\n014\\n015\\n016\\n017\\n018\\n019\\n020\\n021\\n022\\n023\\n024\\n025\\n026\\n027\\n028\\n029\\n030\\n031\\n032\\n033\\n034\\n035\\n036\\n037\\n038\\n039\\n040\\n041\\n042\\n043\\n044\\n045\\n046\\n047\\n048\\n049\\n061\\n062\\n063\\n064\\n065\\n066\\n067\\n068\\n069\\n070\\n071\\n072\\n073\\n074\\n075\\n076\\n077\\n078\\n079\\n080\\n081\\n082\\n083\\n084\\n085\\n086\\n087\\n088\\n089\\n090\\n091\\n092\\n093\\n094\\n095\\n096\\n097\\n098\\n099'}, {'heading': '1 Introduction', 'text': 'Keyphrase or keyword is a piece of short and summative content that expresses the main semantic meaning of a long text. A typical use of keyphrase or keyword is in scientific publications, to provide the core information of a paper. We use

In [4]:
def model_forecasting(model, prompt):
    #print(prompt)
    # Send request to Ollama


    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model, #llama3.2:3b , "qwen3:latest"
            "prompt": prompt, 
            "stream": False, 
            #"think": True,
            # should i include format field?
            "format":{
            "type": "object",
            "properties":{ "prediction": {"type": "string"}, "rationale": {"type":"string"}, "evidence": {"type":"string"} }, 
            "required": ["prediction", "reasoning", "evidence"]
            }
        }
    )
    result = res.json()
    return result


In [5]:
def predict_aspect(pdf_path, review_path, results):
    with open(pdf_path, 'r') as f1:
        paper = json.load(f1) #json file contents for one research paper

    with open(review_path, 'r') as f2:
        review = json.load(f2)


    prompt = build_prompt(paper)
    model = "qwen3:latest"
    output = model_forecasting(model, prompt)
    print(output)
    json_response = json.loads(output.get("response"))
    actual_reviews_list = review.get("reviews")
    actual_reviews_dict = actual_reviews_list[0]

    results[paper.get("name")] = {
        "predicted_score": json_response.get("prediction"),
        "rationale": json_response.get("rationale"),
        "evidence": json_response.get("evidence")
    }
    print(results)
    return results

In [12]:
pdf_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\acl_2017\\train\\parsed_pdfs\\699.pdf.json"
review_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\acl_2017\\train\\reviews\\699.json"
results = {}
results = predict_aspect(pdf_path, review_path, results)

{'model': 'qwen3:latest', 'created_at': '2025-07-08T07:47:48.6632822Z', 'response': '{\n  "prediction": "She is a 4 in the first category, and a 3 in the second. Overall, she is a 3.",\n  "evidence": "The first category is about the soundness of the technical approach. The paper discusses the RNN encoder-decoder model and its applications in NLP, but it does not provide a novel technical approach. It is more of a survey of existing methods. The paper does not propose any new model or method, so the technical approach is not sound as it does not introduce anything new. The second category is about the empirical claims. The paper mentions that the RNN encoder-decoder model has been successful in various NLP tasks, but it does not provide any experiments or results to support these claims. The paper only references previous studies, but does not conduct any experiments to validate the effectiveness of the model. Therefore, the empirical claims are not supported by proper experiments, and 

In [ ]:
print(results)
output_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\dtais_summer\\aspect_review_1_section_prompt2.json"
with open(output_path,'a') as f3:
    json.dump(results,f3)

{'699.pdf': {'predicted_score': "She is not sure about the soundness of the paper, but it's not fatally flawed", 'rationale': None, 'evidence': 'The paper provides a general overview of the RNN Encoder-Decoder model and its improvements. It mentions the introduction of the model, attention mechanism, content copying, and training objective adjustments. However, the paper lacks specific details about the experiments conducted, the results obtained, and their interpretations. The references are to other studies, but the paper itself does not present any original experiments or results. The section on empirical claims is missing, making it difficult to assess the validity of the claims made.'}}


In [86]:
#this function filters for reviews that contain some given aspect score, set as aspect_score_key. needed bc score coverage varies within some directories like iclr_2017
def get_papers_with_score(aspect_score_key, search_dir):
    reviews_with_score = []
    search_dir_list = os.listdir(search_dir)
    for review_name in search_dir_list: 
        #opening each file in the directory to check if it contains the aspect score
        full_path = os.path.join(search_dir, review_name)
        with open(full_path, 'r') as f_open: 
            contents = json.load(f_open)
        contents_list =  contents.get("reviews")
        for review_dict in contents_list: 
            aspect_score = review_dict.get(aspect_score_key)
            if not (aspect_score == None): 
                #aspect score is present so append path
                reviews_with_score.append(full_path)
    return reviews_with_score

In [84]:
iclr_2017_train_reviews = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews"
reviews_with_soundness = get_papers_with_score(aspect_score_key="SOUNDNESS_CORRECTNESS", search_dir = iclr_2017_train_reviews)

In [85]:
print(reviews_with_soundness)
print(len(reviews_with_soundness))

['C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\304.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\304.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\304.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\304.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\305.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\305.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\305.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerR

In [ ]:
def sample_papers_with_score(paths_with_score, seed, size):
    #given a set seed and sample size, returns a random sample of reviews containing the aspect score. 
    random.seed(seed)
    return random.sample(paths_with_score, size)

In [78]:
def get_true_soundness(review_file_path): #returns the average of the soundness scores contained in the human generated reviews
    with open(review_file_path, 'r') as f_review:
        review_file_contents = json.load(f_review)
        
    review_list = review_file_contents.get("reviews")
    score_list = []
    for review_dict in review_list[1:]: 
        soundness = review_dict.get("SOUNDNESS_CORRECTNESS")
        
        if not (soundness == None): 
            #checking if soundness score is present bc not all reviews seem to have it
            score_list.append(int(soundness))
            
    if not score_list:
        print(f'SOUNDNESS_CORRECTNESS not found in {str(review_file_path)}')
        true_avg_score = None #file doesn't have this score
    else: 
        true_avg_score = statistics.mean(score_list)
       
    return true_avg_score

In [15]:
review_file_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\304.json"

In [ ]:
get_true_soundness(review_file_path)



2


2

In [74]:
from sklearn.metrics import root_mean_squared_error

def get_soundness_accuracy(prediction_path, review_file_path):
    #how should i measure accuracy? percent error? st deviation? 
    #PeerRead paper uses root mean square error (not sure why not just MSE but RMSE?), OpenReviewer uses absolute difference
    
    true_scores = []
    predicted_scores = []

    with open(prediction_path, 'r') as f:
        prediction_dict = json.load(f)

    for paper in prediction_dict:
        response_dict = prediction_dict.get(paper)
        predicted_scores.append(response_dict.get("predicted"))
        review_file = str(os.path.splitext(paper)[0])+".json"
        review_full_path = os.path.join(review_file_path,review_file) #need to get full path 
        true_scores.append(get_true_soundness(review_full_path))


    print(true_scores)  
    print(predicted_scores) 
    rmse = root_mean_squared_error(true_scores, predicted_scores)
    print(rmse)
    


In [77]:
prediction_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\dtais_summer\\qwen3_soundness_paper_100_seed_50.json"
review_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\acl_2017\\train\\reviews"
get_soundness_accuracy(prediction_path, review_path)


SOUNDNESS_CORRECTNESS not found in C:\Users\G34371231\OneDrive - The George Washington University\Desktop\PeerRead\data\acl_2017\train\reviews\759.json
SOUNDNESS_CORRECTNESS not found in C:\Users\G34371231\OneDrive - The George Washington University\Desktop\PeerRead\data\acl_2017\train\reviews\564.json
SOUNDNESS_CORRECTNESS not found in C:\Users\G34371231\OneDrive - The George Washington University\Desktop\PeerRead\data\acl_2017\train\reviews\627.json
SOUNDNESS_CORRECTNESS not found in C:\Users\G34371231\OneDrive - The George Washington University\Desktop\PeerRead\data\acl_2017\train\reviews\691.json
SOUNDNESS_CORRECTNESS not found in C:\Users\G34371231\OneDrive - The George Washington University\Desktop\PeerRead\data\acl_2017\train\reviews\343.json
SOUNDNESS_CORRECTNESS not found in C:\Users\G34371231\OneDrive - The George Washington University\Desktop\PeerRead\data\acl_2017\train\reviews\16.json
SOUNDNESS_CORRECTNESS not found in C:\Users\G34371231\OneDrive - The George Washington Un

ValueError: Input contains NaN.